In [1]:
import pandas as pd
import numpy as np
import re
from scipy import stats

import warnings
warnings.filterwarnings("ignore")

In [2]:
cd C:\Users\tr432\Downloads

C:\Users\tr432\Downloads


In [ ]:
# ===============================
# 1) Genus 상대풍부도 DF 만들기: df_taxa (samples x genus)
# ===============================
df0 = pd.read_csv('filter_GutLungAxis_Week2/rel_gg2_genus_table.tsv', sep='\t', skiprows=1)

df0 = df0.replace('__', '_', regex=True)  # g__ -> g_ 로 맞추려는 목적이면 유지
new_cols = df0['#OTU ID'].astype(str).str.split(';', expand=True)
new_cols.columns = ['Kingdom', 'Phylum', 'Class', 'Order', 'Family', 'Genus']
df0 = pd.concat([new_cols, df0], axis=1).drop(columns=['#OTU ID'])

cols = ['Genus'] + df0.columns[6:].tolist()
df_taxa = df0[cols].set_index('Genus').transpose()
df_taxa.index = df_taxa.index.astype(str).str.strip()

# 원하는 genus만
TARGET_GENUS = ['g_Phocaeicola_A', 'g_Bifidobacterium_388775', 'g_Lactobacillus', 'g_Proteus', 'g_Escherichia']

df_taxa = df_taxa[[g for g in TARGET_GENUS if g in df_taxa.columns]]
# Control만 (샘플명이 C로 시작한다고 가정)
df_taxa = df_taxa.loc[df_taxa.index.str.match(r"^C\d{2}", na=False)].copy()  #Edit: 샘플명 규칙성에 맞춰서 변경
print("df_taxa:", df_taxa.shape)
df_taxa

df_taxa: (16, 5)


Genus,g_Phocaeicola_A,g_Bifidobacterium_388775,g_Lactobacillus,g_Proteus,g_Escherichia
C01-2w,0.236873,0.002658,0.007567,0.000000,0.000000
C02-2w,0.188425,0.001864,0.021986,0.000000,0.000000
C03-2w,0.157742,0.003263,0.151334,0.000000,0.000000
C09-2w,0.245171,0.000625,0.096518,0.000000,0.000350
C10-2w,0.249261,0.001187,0.075840,0.000000,0.000178
C11-2w,0.189952,0.000775,0.044625,0.000000,0.000000
V01-2w,0.002149,0.000994,0.000000,0.041889,0.004298
V02-2w,0.001681,0.000733,0.000287,0.001006,0.000000
V03-2w,0.002395,0.000973,0.000210,0.102090,0.013397
V04-2w,0.001727,0.000612,0.000063,0.022737,0.000000


In [46]:
## (선택) 이 부분 옮겨가면서 원하는 데이터만 cytoscape용으로 뽑을 수 있음 (ex. taxa만 or 대사체만)
#df_final = df_taxa.select_dtypes(include=[np.number]).copy()
#df_final

In [47]:
# ===============================
# 0) 기존 데이터프레임 (예: genus+metabolites 붙인 최종 df)
# ===============================
df_main = df_taxa.copy()   # <- 당신이 합쳐둔 기존 DF 이름으로 바꾸세요 (예: df_final)

# 인덱스에서 C01 / V02 같은 베이스 샘플ID 추출
df_main = df_main.copy()
df_main["SampleBase"] = pd.Index(df_main.index.astype(str)).str.extract(r"([CV]\d{2})", expand=False).values

# ===============================
# 1) BM cellcount 로드/정리
# ===============================
bm = pd.read_excel('../Desktop/Roh/_Study_Design/25.05.02~_Gut-Lung_Axis/예비실험1/BM_cellcount_예비실험1.xlsx', sheet_name='파이썬용')
bm = bm.iloc[:, :10]
bm = bm[bm['Date'].isin(['25.09.17', '25.11.05', '25.11.06'])]
bm = bm[bm['Use'] == 'o'].copy()

for c in ['BM_whole', 'BM_wbc', 'b.w.', 'Spleen/b.w.']:
    bm[c] = pd.to_numeric(bm[c].replace('-', np.nan), errors='coerce')

# "샘플명 컬럼" 자동 탐색 (C01, V02 형태가 들어있는 컬럼)
cand = [c for c in bm.columns if bm[c].astype(str).str.contains(r"^[CV]\d{2}", regex=True, na=False).any()]
if len(cand) == 0:
    raise ValueError("BM df에서 C01/V01 같은 샘플명이 들어있는 컬럼을 못 찾음. 해당 컬럼명을 직접 지정하세요.")
sample_col = cand[0]

bm["SampleBase"] = bm[sample_col].astype(str).str.extract(r"([CV]\d{2})", expand=False)

# 같은 SampleBase가 여러 행이면 평균으로 요약(원하면 median으로 바꿔도 됨)
bm_agg = bm.groupby("SampleBase", as_index=True)[["BM_whole", "BM_wbc", 'b.w.', 'Spleen/b.w.']].mean()

# ===============================
# 2) 기존 df_main에 붙이기 (index 유지)
# ===============================
df_merged = df_main.join(bm_agg, on="SampleBase")
df_merged = df_merged.drop(columns=["SampleBase"])

print(df_merged.shape)
df_merged

(16, 9)


,g_Phocaeicola_A,g_Bifidobacterium_388775,g_Lactobacillus,g_Proteus,g_Escherichia,BM_whole,BM_wbc,b.w.,Spleen/b.w.
C01-2w,0.236873,0.002658,0.007567,0.000000,0.000000,NaN,1.964217e+06,22.8,2.807018
C02-2w,0.188425,0.001864,0.021986,0.000000,0.000000,NaN,9.699833e+05,23.4,5.811966
C03-2w,0.157742,0.003263,0.151334,0.000000,0.000000,8.480000e+06,5.680000e+05,22.8,2.973684
C09-2w,0.245171,0.000625,0.096518,0.000000,0.000350,9.883333e+06,1.281367e+06,22.4,3.330357
C10-2w,0.249261,0.001187,0.075840,0.000000,0.000178,1.885000e+06,2.433967e+05,22.4,2.647321
C11-2w,0.189952,0.000775,0.044625,0.000000,0.000000,2.693333e+06,5.725633e+05,22.6,2.526549
V01-2w,0.002149,0.000994,0.000000,0.041889,0.004298,NaN,5.347500e+05,17.9,1.564246
V02-2w,0.001681,0.000733,0.000287,0.001006,0.000000,NaN,8.015833e+05,20.6,2.233010
V03-2w,0.002395,0.000973,0.000210,0.102090,0.013397,NaN,8.561333e+05,13.1,0.839695
V04-2w,0.001727,0.000612,0.000063,0.022737,0.000000,NaN,7.585000e+05,19.5,2.410256


In [48]:
## (선택) 이 부분 옮겨가면서 원하는 데이터만 cytoscape용으로 뽑을 수 있음 (ex. taxa만 or 대사체만)
#df_final = df_merged.select_dtypes(include=[np.number]).copy()
#df_final

In [49]:
# 1) Fecal: (metabolite x sample) -> (sample x metabolite)
fec = pd.read_excel('cytoscape_GutLungAxis/correlation_input용_Fecal_POS.xlsx')
fec = fec.rename(columns={fec.columns[0]: "Name"})  # 첫 컬럼이 Name이라고 가정
fec_mat = fec.set_index("Name").T
fec_mat.index = fec_mat.index.astype(str).str.strip().str.replace("_", "-", regex=False)

# 컬럼명 prefix 정리 (Fecal_ -> Fecal__)
fec_mat.columns = [re.sub(r"^Fecal_", "Fecal__", str(c)).strip() for c in fec_mat.columns]

fec_mat

,"Fecal__4-(2-Amino-3-hydroxyphenyl)-2,4-dioxobutanoic acid",Fecal__5-Hydroxy-DL-tryptophan,Fecal__5-Hydroxyindole-3-acetic acid,Fecal__Acetylcholine,Fecal__Choline,Fecal__dl-3-Indolelactic acid,Fecal__Indole,Fecal__Indole-3-acetaldehyde,Fecal__Indole-3-acetic acid,Fecal__Kynurenic acid,Fecal__Lysolecithin,"Fecal__LysoPC(18:3(9Z,12Z,15Z))",Fecal__LysoPC(P-18:0),Fecal__lysophosphatidylcholine 14:1(9Z)/0:0,Fecal__PC(18:1/0:0),Fecal__Serotonin,Fecal__Tryptamine
C01-2w,7.180968,6.451549,8.605135,7.938758,8.690691,6.597379,6.205441,5.155288,7.961805,8.214161,9.133780,7.646981,7.114535,5.062744,9.056284,5.316448,5.238787
C02-2w,6.770149,6.098269,8.502110,8.020070,8.669099,6.475137,6.383762,5.144792,7.654687,7.955532,9.148601,7.845332,7.122747,6.210531,9.073336,5.151913,5.037841
C03-2w,6.319675,6.262939,8.543580,8.006235,8.578077,6.361804,5.932708,5.512123,6.873976,7.939838,8.988543,7.497662,7.429396,6.008076,8.781301,4.971442,7.128464
C04-2w,7.048525,6.009012,8.435388,7.792101,8.486549,6.508299,5.940839,5.585176,7.572676,7.920016,9.006971,7.719740,6.700044,5.799382,8.912506,4.973846,7.127547
C05-2w,6.811297,6.311617,8.502851,7.800251,8.467901,6.423230,6.294984,6.265564,7.497015,8.105681,8.916402,7.535162,6.932294,5.832327,8.820754,4.927698,7.076979
C06-2w,6.902966,6.342036,8.531457,7.999687,8.809243,6.636629,6.439278,6.303453,7.072403,8.008802,9.282892,7.809066,7.991633,6.252450,9.144950,5.128030,6.790318
C07-2w,7.310477,6.439897,8.534058,7.929274,8.546291,6.553283,6.183734,6.341120,7.354547,8.123892,9.101972,7.636789,7.124544,5.985388,8.955609,5.151551,7.086653
C08-2w,5.960235,6.493899,8.309473,7.955052,8.241496,6.315549,6.185901,6.291266,6.726549,7.735613,8.740516,7.196400,7.033373,5.333082,8.523991,5.140523,7.466680
C09-2w,6.453843,6.267790,8.556515,8.115241,8.323642,6.439692,6.445527,5.727902,7.221789,8.106748,9.078233,7.811445,6.843214,6.066623,8.998617,4.986522,7.066768
C10-2w,6.288132,6.165273,8.488902,8.017740,8.299452,6.509790,5.742584,5.304215,7.063625,8.111634,8.955156,7.560425,6.665679,6.036627,8.747424,5.090549,6.878368


In [50]:
# 2) Serum: (metabolite x sample) -> (sample x metabolite)
ser = pd.read_excel('cytoscape_GutLungAxis/correlation_input용_Serum_POS.xlsx')
ser = ser.rename(columns={ser.columns[0]: "Name"})
ser_mat = ser.set_index("Name").T
ser_mat.index = ser_mat.index.astype(str).str.strip().str.replace("_", "-", regex=False)

# 컬럼명 prefix 정리 (Serum_ -> Serum__)
ser_mat.columns = [re.sub(r"^Serum_", "Serum__", str(c)).strip() for c in ser_mat.columns]

ser_mat

,Serum__Lysolecithin,Serum__PC(18:1/0:0),"Serum__lysophosphatidylethanolamine (22:6(4Z,7Z,10Z,13Z,16Z,19Z)/0:0)",Serum__Methyl indole-3-acetate,Serum__Indole-3-acetic acid,Serum__Tryptophol
C03-2w,9.202780,8.858998,7.587120,6.737196,6.229550,5.658972
C04-2w,9.294801,8.878034,7.648191,6.835379,6.346580,6.017729
C05-2w,9.223403,8.747176,7.596997,6.674265,6.447492,5.790328
C06-2w,9.363768,8.915922,7.800353,6.874966,6.452933,5.519784
C07-2w,9.147197,8.669846,7.518602,6.711249,6.342866,5.705078
C08-2w,9.304696,8.788235,7.727631,6.874105,6.441530,5.679409
C09-2w,9.273235,8.732469,7.658570,6.711743,6.611292,5.371476
C10-2w,9.341150,8.897697,7.692782,6.810104,6.275883,5.935567
C11-2w,9.325066,8.879147,7.692535,6.846916,6.610359,5.928088
V05-2w,9.148001,8.497972,7.559696,5.169697,5.750790,5.845376


In [52]:
# 3) df_merged(행=샘플) 에 붙이기
df_final = df_merged.copy()
#df_final = fec_mat.copy()
df_final.index = df_final.index.astype(str).str.strip()  # 혹시 모를 공백 제거

df_final = df_final.join(fec_mat, how="left")
df_final = df_final.join(ser_mat, how="left")

print("df_final:", df_final.shape)

## (선택) 저장
#df_final.to_csv("merged_correlation_input용_merged.csv", index=True, encoding="utf-8-sig")
df_final

df_final: (16, 32)


,g_Phocaeicola_A,g_Bifidobacterium_388775,g_Lactobacillus,g_Proteus,g_Escherichia,BM_whole,BM_wbc,b.w.,Spleen/b.w.,"Fecal__4-(2-Amino-3-hydroxyphenyl)-2,4-dioxobutanoic acid",...,Fecal__lysophosphatidylcholine 14:1(9Z)/0:0,Fecal__PC(18:1/0:0),Fecal__Serotonin,Fecal__Tryptamine,Serum__Lysolecithin,Serum__PC(18:1/0:0),"Serum__lysophosphatidylethanolamine (22:6(4Z,7Z,10Z,13Z,16Z,19Z)/0:0)",Serum__Methyl indole-3-acetate,Serum__Indole-3-acetic acid,Serum__Tryptophol
C01-2w,0.236873,0.002658,0.007567,0.000000,0.000000,NaN,1.964217e+06,22.8,2.807018,7.180968,...,5.062744,9.056284,5.316448,5.238787,NaN,NaN,NaN,NaN,NaN,NaN
C02-2w,0.188425,0.001864,0.021986,0.000000,0.000000,NaN,9.699833e+05,23.4,5.811966,6.770149,...,6.210531,9.073336,5.151913,5.037841,NaN,NaN,NaN,NaN,NaN,NaN
C03-2w,0.157742,0.003263,0.151334,0.000000,0.000000,8.480000e+06,5.680000e+05,22.8,2.973684,6.319675,...,6.008076,8.781301,4.971442,7.128464,9.202780,8.858998,7.587120,6.737196,6.229550,5.658972
C09-2w,0.245171,0.000625,0.096518,0.000000,0.000350,9.883333e+06,1.281367e+06,22.4,3.330357,6.453843,...,6.066623,8.998617,4.986522,7.066768,9.273235,8.732469,7.658570,6.711743,6.611292,5.371476
C10-2w,0.249261,0.001187,0.075840,0.000000,0.000178,1.885000e+06,2.433967e+05,22.4,2.647321,6.288132,...,6.036627,8.747424,5.090549,6.878368,9.341150,8.897697,7.692782,6.810104,6.275883,5.935567
C11-2w,0.189952,0.000775,0.044625,0.000000,0.000000,2.693333e+06,5.725633e+05,22.6,2.526549,6.577554,...,5.989205,9.069026,5.071255,7.160475,9.325066,8.879147,7.692535,6.846916,6.610359,5.928088
V01-2w,0.002149,0.000994,0.000000,0.041889,0.004298,NaN,5.347500e+05,17.9,1.564246,7.342271,...,8.526715,9.073663,6.166574,5.486710,NaN,NaN,NaN,NaN,NaN,NaN
V02-2w,0.001681,0.000733,0.000287,0.001006,0.000000,NaN,8.015833e+05,20.6,2.233010,7.156902,...,8.185271,9.097813,6.788342,5.378308,NaN,NaN,NaN,NaN,NaN,NaN
V03-2w,0.002395,0.000973,0.000210,0.102090,0.013397,NaN,8.561333e+05,13.1,0.839695,7.395055,...,8.110519,9.161982,7.457132,5.777872,NaN,NaN,NaN,NaN,NaN,NaN
V04-2w,0.001727,0.000612,0.000063,0.022737,0.000000,NaN,7.585000e+05,19.5,2.410256,7.272275,...,8.365951,9.124754,7.301876,5.674947,NaN,NaN,NaN,NaN,NaN,NaN


In [53]:
df_final = df_final.select_dtypes(include=[np.number])
df_final

,g_Phocaeicola_A,g_Bifidobacterium_388775,g_Lactobacillus,g_Proteus,g_Escherichia,BM_whole,BM_wbc,b.w.,Spleen/b.w.,"Fecal__4-(2-Amino-3-hydroxyphenyl)-2,4-dioxobutanoic acid",...,Fecal__lysophosphatidylcholine 14:1(9Z)/0:0,Fecal__PC(18:1/0:0),Fecal__Serotonin,Fecal__Tryptamine,Serum__Lysolecithin,Serum__PC(18:1/0:0),"Serum__lysophosphatidylethanolamine (22:6(4Z,7Z,10Z,13Z,16Z,19Z)/0:0)",Serum__Methyl indole-3-acetate,Serum__Indole-3-acetic acid,Serum__Tryptophol
C01-2w,0.236873,0.002658,0.007567,0.000000,0.000000,NaN,1.964217e+06,22.8,2.807018,7.180968,...,5.062744,9.056284,5.316448,5.238787,NaN,NaN,NaN,NaN,NaN,NaN
C02-2w,0.188425,0.001864,0.021986,0.000000,0.000000,NaN,9.699833e+05,23.4,5.811966,6.770149,...,6.210531,9.073336,5.151913,5.037841,NaN,NaN,NaN,NaN,NaN,NaN
C03-2w,0.157742,0.003263,0.151334,0.000000,0.000000,8.480000e+06,5.680000e+05,22.8,2.973684,6.319675,...,6.008076,8.781301,4.971442,7.128464,9.202780,8.858998,7.587120,6.737196,6.229550,5.658972
C09-2w,0.245171,0.000625,0.096518,0.000000,0.000350,9.883333e+06,1.281367e+06,22.4,3.330357,6.453843,...,6.066623,8.998617,4.986522,7.066768,9.273235,8.732469,7.658570,6.711743,6.611292,5.371476
C10-2w,0.249261,0.001187,0.075840,0.000000,0.000178,1.885000e+06,2.433967e+05,22.4,2.647321,6.288132,...,6.036627,8.747424,5.090549,6.878368,9.341150,8.897697,7.692782,6.810104,6.275883,5.935567
C11-2w,0.189952,0.000775,0.044625,0.000000,0.000000,2.693333e+06,5.725633e+05,22.6,2.526549,6.577554,...,5.989205,9.069026,5.071255,7.160475,9.325066,8.879147,7.692535,6.846916,6.610359,5.928088
V01-2w,0.002149,0.000994,0.000000,0.041889,0.004298,NaN,5.347500e+05,17.9,1.564246,7.342271,...,8.526715,9.073663,6.166574,5.486710,NaN,NaN,NaN,NaN,NaN,NaN
V02-2w,0.001681,0.000733,0.000287,0.001006,0.000000,NaN,8.015833e+05,20.6,2.233010,7.156902,...,8.185271,9.097813,6.788342,5.378308,NaN,NaN,NaN,NaN,NaN,NaN
V03-2w,0.002395,0.000973,0.000210,0.102090,0.013397,NaN,8.561333e+05,13.1,0.839695,7.395055,...,8.110519,9.161982,7.457132,5.777872,NaN,NaN,NaN,NaN,NaN,NaN
V04-2w,0.001727,0.000612,0.000063,0.022737,0.000000,NaN,7.585000e+05,19.5,2.410256,7.272275,...,8.365951,9.124754,7.301876,5.674947,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
import pandas as pd
import numpy as np
import re
from scipy import stats

import warnings
warnings.filterwarnings("ignore")

In [16]:
cd C:\Users\tr432\Downloads\Metabolomics_GutLungAxis_re-pipeline\merged_correlation

C:\Users\tr432\Downloads\Metabolomics_GutLungAxis_re-pipeline\merged_correlation


In [17]:
df_final = pd.read_excel('merged_corr_input.xlsx', sheet_name='merged_co-occurrence', index_col=[0])
df_final

,g_Bacteroides_H_857956,g_Bifidobacterium_388775,g_Escherichia,g_Lactobacillus,g_Phocaeicola_A,g_Prevotella,g_Proteus,Lactobacillus jensenii,Bacillus subtilis,Staphylococcus aureus,...,[Serum] MIA,[Serum] LysoPC,[Serum] PC(18:1/0:0),[Serum] Choline,"[Serum] LysoPE (22:6(4Z,7Z,10Z,13Z,16Z,19Z)/0:0)","[Serum] LysoPC(20:5(5Z,8Z,11Z,14Z,17Z))","[Serum] LysoPC(18:3(9Z,12Z,15Z))",f_Enterobacteriaceae_A_725029,f_Enterobacteriaceae_A_732334,Enterobacteriaceae
Features,,,,,,,,,,,,,,,,,,,,,
C01-2w,0.012748,0.002658,0.000000,0.007567,0.236873,0.000000,0.000000,0.00001,0.00045,0.00024,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.002030,0.000000,0.00543
C02-2w,0.007159,0.001864,0.000000,0.021986,0.188425,0.000000,0.000000,0.00001,0.00020,0.00024,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.002150,0.000000,0.00402
C03-2w,0.010697,0.003263,0.000000,0.151334,0.157742,0.000000,0.000000,0.00001,0.00016,0.00028,...,1.281325,0.228524,0.525424,0.274534,0.237132,0.588900,0.296641,0.000891,0.000000,0.00463
C04-2w,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.275238,0.180054,0.327473,0.225754,0.082005,-0.041327,-0.107236,NaN,NaN,NaN
C05-2w,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.205308,0.112347,0.145407,0.092915,0.075047,-0.299198,-0.101947,NaN,NaN,NaN
C06-2w,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.246829,0.130613,0.222014,0.059133,0.303176,0.146891,0.206516,NaN,NaN,NaN
C07-2w,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.243155,-0.047752,0.025428,0.429223,-0.106568,-0.180310,-0.233800,NaN,NaN,NaN
C08-2w,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.261042,0.023175,-0.009685,-0.316325,0.141071,-0.374501,-0.097360,NaN,NaN,NaN
C09-2w,0.011463,0.000625,0.000350,0.096518,0.245171,0.000000,0.000000,0.00001,0.00011,0.00020,...,1.135807,-0.183702,-0.215762,-0.303271,-0.208510,-0.574272,-0.221101,0.001508,0.000000,0.00243


In [18]:
from scipy.stats import shapiro

# 각 컬럼에 대해 Shapiro-Wilk 테스트 수행 -> 0.05보다 작으면 정규성 x
results = {}
numeric_columns = []  # 수치형 컬럼 이름 저장
for column in df_final.columns:
    if pd.api.types.is_numeric_dtype(df_final[column]):  # 컬럼이 수치형 데이터인지 확인
        data = df_final[column].dropna()
        if data.nunique() > 1:  # 데이터가 최소 두 개의 고유값을 가져야 함
            stat, p_value = shapiro(data)
            results[column] = {'Statistic': stat, 'p-value': p_value}
            numeric_columns.append(column)  # 수치형 컬럼 이름 추가
        else:
            print(f"Column '{column}' has no variance. Skipping Shapiro-Wilk test.")

# 결과를 데이터프레임으로 변환
results_df = pd.DataFrame.from_dict(results, orient='index').reset_index()
results_df = results_df.rename(columns={'index': 'Column'})


results_df.to_csv('merged_shapiro.csv', index=False, encoding='utf-8-sig')
results_df

,Column,Statistic,p-value
0,g_Bacteroides_H_857956,0.774081,1.258140e-03
1,g_Bifidobacterium_388775,0.822204,5.454452e-03
2,g_Escherichia,0.440285,7.057464e-07
3,g_Lactobacillus,0.641553,4.104349e-05
4,g_Phocaeicola_A,0.690693,1.335076e-04
...,...,...,...
60,"[Serum] LysoPC(20:5(5Z,8Z,11Z,14Z,17Z))",0.939309,3.737584e-01
61,"[Serum] LysoPC(18:3(9Z,12Z,15Z))",0.814420,5.663611e-03
62,f_Enterobacteriaceae_A_725029,0.731337,3.819501e-04
63,f_Enterobacteriaceae_A_732334,0.465184,1.106173e-06


In [19]:
# ===============================
# 1) Spearman rho/p 행렬 (pairwise, NaN 자동 처리)
# ===============================
def spearman_rho_p(df: pd.DataFrame):
    cols = df.columns.astype(str).tolist()
    n = len(cols)
    rho = np.eye(n, dtype=float)
    p   = np.ones((n, n), dtype=float)

    for i in range(n):
        for j in range(i+1, n):
            r, pv = stats.spearmanr(df.iloc[:, i], df.iloc[:, j], nan_policy="omit")
            rho[i, j] = rho[j, i] = r
            p[i, j]   = p[j, i]   = pv

    rho_df = pd.DataFrame(rho, index=cols, columns=cols)
    p_df   = pd.DataFrame(p,   index=cols, columns=cols)
    return rho_df, p_df

rho_mat, p_vals = spearman_rho_p(df_final)

p_vals.to_csv('merged_pval_spearman.csv', index=True, encoding="utf-8-sig")
rho_mat.to_csv('merged_rho_spearman.csv', index=True, encoding="utf-8-sig")
p_vals

,g_Bacteroides_H_857956,g_Bifidobacterium_388775,g_Escherichia,g_Lactobacillus,g_Phocaeicola_A,g_Prevotella,g_Proteus,Lactobacillus jensenii,Bacillus subtilis,Staphylococcus aureus,...,[Serum] MIA,[Serum] LysoPC,[Serum] PC(18:1/0:0),[Serum] Choline,"[Serum] LysoPE (22:6(4Z,7Z,10Z,13Z,16Z,19Z)/0:0)","[Serum] LysoPC(20:5(5Z,8Z,11Z,14Z,17Z))","[Serum] LysoPC(18:3(9Z,12Z,15Z))",f_Enterobacteriaceae_A_725029,f_Enterobacteriaceae_A_732334,Enterobacteriaceae
g_Bacteroides_H_857956,1.000000,0.000890,0.227109,0.000050,0.000005,0.160035,0.602106,0.000188,0.000029,0.001671,...,0.003406,0.802181,0.250496,0.946840,0.920318,0.578406,0.283003,1.239331e-03,0.602106,1.960240e-03
g_Bifidobacterium_388775,0.000890,1.000000,0.367962,0.044659,0.002743,0.303894,0.886941,0.113444,0.015109,0.007736,...,0.039843,0.227552,0.111088,0.111088,0.737938,0.725250,0.638259,5.824671e-02,0.886941,1.372880e-01
g_Escherichia,0.227109,0.367962,1.000000,0.224819,0.344645,0.020337,0.598854,0.113314,0.137382,0.306788,...,0.043164,0.399450,0.918044,0.918044,0.184806,0.152707,0.032116,3.187911e-02,0.598854,1.150742e-01
g_Lactobacillus,0.000050,0.044659,0.224819,1.000000,0.000448,0.249654,0.182910,0.002336,0.000478,0.002076,...,0.008931,0.444399,0.019528,0.737938,0.815172,0.802181,0.815172,1.193198e-03,0.182910,2.448527e-03
g_Phocaeicola_A,0.000005,0.002743,0.344645,0.000448,1.000000,0.114759,0.557876,0.008089,0.000412,0.001292,...,0.007488,0.828717,0.186905,0.960240,0.986743,0.445838,0.310376,3.024719e-03,0.557876,1.912964e-03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"[Serum] LysoPC(20:5(5Z,8Z,11Z,14Z,17Z))",0.578406,0.725250,0.152707,0.802181,0.445838,0.369116,1.000000,0.638240,0.888697,0.710288,...,0.499641,0.001281,0.048004,0.177342,0.076070,1.000000,0.000135,3.465935e-01,1.000000,1.704707e-01
"[Serum] LysoPC(18:3(9Z,12Z,15Z))",0.283003,0.638259,0.032116,0.815172,0.310376,0.118975,1.000000,0.638240,0.538609,0.308862,...,0.332053,0.012080,0.101361,0.959700,0.051725,0.000135,1.000000,8.972403e-02,1.000000,1.116330e-01
f_Enterobacteriaceae_A_725029,0.001239,0.058247,0.031879,0.001193,0.003025,0.011916,0.197545,0.000192,0.000222,0.000024,...,0.000408,0.854813,0.467089,0.676065,0.651477,0.346594,0.089724,1.000000e+00,0.197545,2.273675e-08
f_Enterobacteriaceae_A_732334,0.602106,0.886941,0.598854,0.182910,0.557876,0.975796,1.000000,0.118999,0.080442,0.121666,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.975448e-01,1.000000,2.655526e-01


In [20]:
# ===============================
# 2) BKY(q-value) 보정 (upper triangle만 대상으로 FDR)
#    statsmodels method='fdr_tsbky' = BKY(two-stage)
# ===============================
from statsmodels.stats.multitest import multipletests

mask = np.triu(np.ones(p_vals.shape, dtype=bool), k=1)

p_stack = p_vals.where(mask).stack()          # (source,target)->p
p_vec   = p_stack.values.astype(float)

q_vec = multipletests(p_vec, alpha=ALPHA, method="fdr_tsbky")[1]

q_mat = pd.DataFrame(np.ones(p_vals.shape), index=p_vals.index, columns=p_vals.columns)
for (src, tgt), q in zip(p_stack.index, q_vec):
    q_mat.loc[src, tgt] = q
    q_mat.loc[tgt, src] = q
np.fill_diagonal(q_mat.values, 0.0)

q_mat.to_csv('merged_qval_BKY_spearman.csv', index=True, encoding="utf-8-sig")
q_mat

,g_Bacteroides_H_857956,g_Bifidobacterium_388775,g_Escherichia,g_Lactobacillus,g_Phocaeicola_A,g_Prevotella,g_Proteus,Lactobacillus jensenii,Bacillus subtilis,Staphylococcus aureus,...,[Serum] MIA,[Serum] LysoPC,[Serum] PC(18:1/0:0),[Serum] Choline,"[Serum] LysoPE (22:6(4Z,7Z,10Z,13Z,16Z,19Z)/0:0)","[Serum] LysoPC(20:5(5Z,8Z,11Z,14Z,17Z))","[Serum] LysoPC(18:3(9Z,12Z,15Z))",f_Enterobacteriaceae_A_725029,f_Enterobacteriaceae_A_732334,Enterobacteriaceae
g_Bacteroides_H_857956,0.000000,0.007666,0.320790,0.001110,0.000281,0.255590,0.600232,0.002892,0.000818,0.011684,...,0.017140,0.717007,0.341679,0.790872,0.779239,0.586786,0.371968,0.009675,0.600232,0.012693
g_Bifidobacterium_388775,0.007666,0.000000,0.444947,0.103443,0.015173,0.390817,0.761731,0.197822,0.047446,0.030053,...,0.094499,0.320790,0.196570,0.196570,0.682223,0.674216,0.622806,0.125878,0.761731,0.228134
g_Escherichia,0.320790,0.444947,0.000000,0.319636,0.425219,0.058742,0.600232,0.197822,0.228134,0.393025,...,0.100569,0.467298,0.778892,0.778892,0.280917,0.246956,0.079839,0.079486,0.600232,0.199210
g_Lactobacillus,0.001110,0.103443,0.319636,0.000000,0.004919,0.341014,0.278796,0.013846,0.005117,0.012986,...,0.033153,0.498919,0.057033,0.682223,0.722179,0.717007,0.722179,0.009439,0.278796,0.014197
g_Phocaeicola_A,0.000281,0.015173,0.425219,0.004919,0.000000,0.198871,0.575214,0.031064,0.004583,0.009904,...,0.029576,0.729601,0.283077,0.799666,0.803149,0.500198,0.396340,0.015935,0.575214,0.012629
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"[Serum] LysoPC(20:5(5Z,8Z,11Z,14Z,17Z))",0.586786,0.674216,0.246956,0.717007,0.500198,0.444947,0.803149,0.622806,0.761731,0.665492,...,0.536768,0.009872,0.109091,0.272297,0.150922,0.000000,0.002220,0.426678,0.803149,0.265901
"[Serum] LysoPC(18:3(9Z,12Z,15Z))",0.371968,0.622806,0.079839,0.722179,0.396340,0.204310,0.803149,0.622806,0.564121,0.395076,...,0.414273,0.040216,0.184654,0.799614,0.115802,0.002220,0.000000,0.168478,0.803149,0.196718
f_Enterobacteriaceae_A_725029,0.009675,0.125878,0.079486,0.009439,0.015935,0.039972,0.292703,0.002892,0.003147,0.000782,...,0.004583,0.744918,0.516410,0.646480,0.629454,0.426678,0.168478,0.000000,0.292703,0.000007
f_Enterobacteriaceae_A_732334,0.600232,0.761731,0.600232,0.278796,0.575214,0.803149,0.803149,0.204310,0.157357,0.206689,...,0.803149,0.803149,0.803149,0.803149,0.803149,0.803149,0.803149,0.292703,0.000000,0.353481


In [22]:
ALPHA   = 0.05
THRESH  = 0.4  #Edit: |rho| threshold -> pval이랑 rho 결과 보고 조정

In [23]:
# ===============================
# 3) Cytoscape edges: p or q<0.05 & |rho|>=0.x
# ===============================
rho_stack = rho_mat.where(mask).stack()

edges = pd.DataFrame({
    "source": rho_stack.index.get_level_values(0),
    "target": rho_stack.index.get_level_values(1),
    "rho":    rho_stack.values.astype(float),
    "p":      p_stack.values.astype(float),
    "q":      q_vec
})
edges["abs_rho"] = edges["rho"].abs()
print(f"Removed by |rho| < {THRESH}:", (edges["abs_rho"] < THRESH).sum())
print(f"Removed by p >= {ALPHA}:", (edges["p"] >= ALPHA).sum())

edges = edges.loc[(edges["p"] <= ALPHA) & (edges["abs_rho"] >= THRESH)].copy()
edges["sign"] = np.where(edges["rho"] >= 0, "pos", "neg")
edges["interaction"] = "spearman"

edges = edges[["source", "target", "interaction", "rho", "abs_rho", "p", "sign"]]

edges.to_csv('cytoscape_edges.csv', index=False, encoding="utf-8-sig")
print("edges:", len(edges))

Removed by |rho| < 0.4: 1052
Removed by p >= 0.05: 1336
edges: 744


In [24]:
# ===============================
# 4) Cytoscape nodes
# ===============================
nodes = pd.unique(pd.concat([edges["source"], edges["target"]], ignore_index=True))
node_df = pd.DataFrame({"id": nodes, "type": ""})  # type은 엑셀에서 채우기

node_df.to_csv('cytoscape_nodes.csv', index=False, encoding="utf-8-sig")
print("nodes:", len(node_df))

nodes: 64
